# Imports

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [57]:
import warnings
warnings.filterwarnings('ignore')


# Loading Data

## Downloading Data

In [3]:
!wget https://raw.githubusercontent.com/Hari31416/Portfolio/main/ML/Disaster_Tweets/data/train_clean.csv
!wget https://raw.githubusercontent.com/Hari31416/Portfolio/main/ML/Disaster_Tweets/data/test_clean.csv

--2022-06-17 04:33:25--  https://raw.githubusercontent.com/Hari31416/Portfolio/main/ML/Disaster_Tweets/data/train_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 687237 (671K) [text/plain]
Saving to: ‘train_clean.csv’

train_clean.csv     100%[===================>] 671.13K  --.-KB/s    in 0.04s   

2022-06-17 04:33:26 (15.4 MB/s) - ‘train_clean.csv’ saved [687237/687237]

--2022-06-17 04:33:26--  https://raw.githubusercontent.com/Hari31416/Portfolio/main/ML/Disaster_Tweets/data/test_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting respons

## Reading and Making Data Ready

In [4]:
train = pd.read_csv('train_clean.csv')
test = pd.read_csv('test_clean.csv')

In [5]:
id = test["id"]

In [6]:
train_text = train["text"]
test_text = test["text"]
target = train["target"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(train_text, target, test_size=0.1, random_state=42)

# Modeling

## Some Functions

In [8]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall and f1 score of a binary classification model.

  Args:
  -----
  y_true = true labels in the form of a 1D array
  y_pred = predicted labels in the form of a 1D array

  Returns a dictionary of accuracy, precision, recall, f1-score.
  """
  # Calculate model accuracy
  model_accuracy = accuracy_score(y_true, y_pred) * 100
  # Calculate model precision, recall and f1 score using "weighted" average
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  model_results = {"accuracy": model_accuracy,
                  "precision": model_precision,
                  "recall": model_recall,
                  "f1": model_f1}
  return model_results

In [40]:
def create_tensorboard_callback(experiment_name, dir_name="mnist"):
  log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
  tensorboard_callback = tf.keras.callbacks.TensorBoard(
      log_dir=log_dir
  )
  print(f"Saving TensorBoard log files to: {log_dir}")
  return tensorboard_callback

def create_early_stopping_callback(monitor = 'val_accuracy', patience=5):
  early_stopping_callback = tf.keras.callbacks.EarlyStopping(
      monitor=monitor, patience=patience
  )
  return early_stopping_callback

def create_model_checkpoint(experiment_name, dir_name="tweets",
                            save_freq='epoch', monitor = "val_accuracy"):

  checkpoint_dir = dir_name + "/" + experiment_name
  checkpoint_callback = ModelCheckpoint(
    checkpoint_dir, monitor=monitor, verbose=1, save_best_only=True,
    save_weights_only=False, mode='auto', save_freq=save_freq,
  )
  print(f"Saving model checkpoint to: {checkpoint_dir}")
  return checkpoint_callback

## Base Model (Naive Bayes)

In [9]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("nb", MultinomialNB())
])
pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('nb', MultinomialNB())])

In [10]:
pipe.score(X_train, y_train), pipe.score(X_test, y_test)

(0.8982481751824818, 0.7913385826771654)

The base model is giving an accuracy of 79%.

In [11]:
base_model_performance = calculate_results(y_test, pipe.predict(X_test))
base_model_performance

{'accuracy': 79.13385826771653,
 'f1': 0.7865535576835297,
 'precision': 0.7974211556888723,
 'recall': 0.7913385826771654}

## Deep Learning Models

### Text Vectorization and Embedding

In [12]:
MAX_VOCAB_SIZE = 10000
MAX_LENGTH = 20
tokenizer = tfl.TextVectorization(max_tokens=MAX_VOCAB_SIZE, output_sequence_length=MAX_LENGTH, name="tokenizer")
tokenizer.adapt(train_text)

In [13]:
vocab = tokenizer.get_vocabulary()
vocab[:10], vocab[-10:]

(['', '[UNK]', 'like', 'e', 'amp', 'im', 'a', 'fire', 't', 'get'],
 ['nailreal',
  'nagaski',
  'naemolgo',
  'nades',
  'naayf',
  'naaa',
  'na',
  'n36',
  'n15b',
  'myths'])

In [14]:
embedd = tfl.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH, name="embedd_1")

### Creating Datasets

In [30]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

In [31]:
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

### Model 1

We'll start with a small model.

In [32]:
input = tfl.Input(shape=(1,), dtype="string", name="input1")
x = tokenizer(input)
x = embedd(x)
x = tfl.Flatten(name="flatten1")(x) 
x = tfl.Dense(128, activation="relu", name="dense1")(x)
output = tfl.Dense(1, activation="sigmoid")(x)
model_1 = Model(inputs=input, outputs=output)
model_1.compile(optimizer="adam", loss="binary_crossentropy", metrics = ["accuracy"])
model_1.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input1 (InputLayer)         [(None, 1)]               0         
                                                                 
 tokenizer (TextVectorizatio  (None, 20)               0         
 n)                                                              
                                                                 
 embedd_1 (Embedding)        (None, 20, 128)           1280000   
                                                                 
 flatten1 (Flatten)          (None, 2560)              0         
                                                                 
 dense1 (Dense)              (None, 128)               327808    
                                                                 
 dense_9 (Dense)             (None, 1)                 129       
                                                           

In [33]:
history_1 = model_1.fit(train_dataset,
                        epochs=10,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        )

Epoch 1/10
215/215 [==============================] - 1s 5ms/step - loss: 0.0749 - accuracy: 0.9728 - val_loss: 0.7744 - val_accuracy: 0.7612
Epoch 2/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0299 - accuracy: 0.9851 - val_loss: 0.9334 - val_accuracy: 0.7572
Epoch 3/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0269 - accuracy: 0.9857 - val_loss: 1.0584 - val_accuracy: 0.7559
Epoch 4/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0258 - accuracy: 0.9855 - val_loss: 1.1475 - val_accuracy: 0.7572
Epoch 5/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0253 - accuracy: 0.9858 - val_loss: 1.2092 - val_accuracy: 0.7572
Epoch 6/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0249 - accuracy: 0.9861 - val_loss: 1.2625 - val_accuracy: 0.7598
Epoch 7/10
215/215 [==============================] - 1s 4ms/step - loss: 0.0248 - accuracy: 0.9864 - val_loss: 1.3058 - val_accuracy: 0.7493
Epoch 

The model is quickly overfitting.

### Transfer Learning

#### Conv Model

Let's use the USE encoder from tensorflow hub. Let's use a `Conv1D` layer with this encoder.

In [34]:
import tensorflow_hub as hub
# We can use this encoding layer in place of our text_vectorizer and embedding layer
sentence_encoder_layer = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
                                        input_shape=[], # shape of inputs coming to our model 
                                        dtype=tf.string, # data type of inputs coming to the USE layer
                                        trainable=False, # keep the pretrained weights (we'll create a feature extractor)
                                        name="USE") 

In [56]:
# Create model using the Sequential API
model_2 = tf.keras.Sequential([
  sentence_encoder_layer, # take in sentences and then encode them into an embedding
  tfl.Reshape((64, 8), input_shape=(512,)),

  tfl.Conv1D(32, 5, padding="valid"),
  tfl.Conv1D(64, 5, padding="valid"),
   tfl.MaxPool1D(2),
  tfl.Dropout(0.7),
  tfl.BatchNormalization(),

  tfl.Conv1D(128, 5, padding="valid"),
  tfl.Conv1D(256, 5, padding="valid"),
   tfl.MaxPool1D(2),
  tfl.Dropout(0.7),
  tfl.BatchNormalization(),

  tfl.Conv1D(256, 3, padding="valid"),
  tfl.Conv1D(512, 3, padding="valid"),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),
  tfl.Dense(128, activation="relu"),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),
    tfl.GlobalAveragePooling1D(),
  tfl.Dense(1, activation="sigmoid")
], name="model_2_USE")

# Compile model
model_2.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

model_2.summary()

Model: "model_2_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 reshape_9 (Reshape)         (None, 64, 8)             0         
                                                                 
 conv1d_37 (Conv1D)          (None, 60, 32)            1312      
                                                                 
 conv1d_38 (Conv1D)          (None, 56, 64)            10304     
                                                                 
 max_pooling1d_18 (MaxPoolin  (None, 28, 64)           0         
 g1D)                                                            
                                                                 
 dropout_31 (Dropout)        (None, 28, 64)            0         
                                                       

In [58]:
history_2 = model_2.fit(train_dataset,
                        epochs=10,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        )

Epoch 1/10
215/215 [==============================] - 8s 20ms/step - loss: 0.6544 - accuracy: 0.6232 - val_loss: 0.6550 - val_accuracy: 0.5682
Epoch 2/10
215/215 [==============================] - 4s 17ms/step - loss: 0.5841 - accuracy: 0.7066 - val_loss: 0.5830 - val_accuracy: 0.6719
Epoch 3/10
215/215 [==============================] - 4s 18ms/step - loss: 0.5414 - accuracy: 0.7387 - val_loss: 0.4887 - val_accuracy: 0.7835
Epoch 4/10
215/215 [==============================] - 4s 17ms/step - loss: 0.5248 - accuracy: 0.7472 - val_loss: 0.4579 - val_accuracy: 0.8005
Epoch 5/10
215/215 [==============================] - 4s 18ms/step - loss: 0.5028 - accuracy: 0.7655 - val_loss: 0.4517 - val_accuracy: 0.8084
Epoch 6/10
215/215 [==============================] - 4s 18ms/step - loss: 0.5049 - accuracy: 0.7679 - val_loss: 0.4459 - val_accuracy: 0.8071
Epoch 7/10
215/215 [==============================] - 4s 18ms/step - loss: 0.4883 - accuracy: 0.7790 - val_loss: 0.4428 - val_accuracy: 0.8136

Looks promising. Let's fit for some more epochs with callbacks.

In [59]:
history_3 = model_2.fit(train_dataset,
                        epochs=50,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        callbacks = [create_model_checkpoint("conv")]
                        )

Saving model checkpoint to: tweets/conv
Epoch 1/50
215/215 [==============================] - ETA: 0s - loss: 0.4741 - accuracy: 0.7857
Epoch 1: val_accuracy improved from -inf to 0.78346, saving model to tweets/conv


INFO:tensorflow:Assets written to: tweets/conv/assets


INFO:tensorflow:Assets written to: tweets/conv/assets


215/215 [==============================] - 16s 77ms/step - loss: 0.4741 - accuracy: 0.7857 - val_loss: 0.4932 - val_accuracy: 0.7835
Epoch 2/50
212/215 [============================>.] - ETA: 0s - loss: 0.4728 - accuracy: 0.7883
Epoch 2: val_accuracy improved from 0.78346 to 0.81365, saving model to tweets/conv


INFO:tensorflow:Assets written to: tweets/conv/assets


...

INFO:tensorflow:Assets written to: tweets/conv/assets


215/215 [==============================] - 15s 69ms/step - loss: 0.4605 - accuracy: 0.7920 - val_loss: 0.4296 - val_accuracy: 0.8189
Epoch 5/50
214/215 [============================>.] - ETA: 0s - loss: 0.4644 - accuracy: 0.7937
Epoch 5: val_accuracy did not improve from 0.81890
215/215 [==============================] - 4s 18ms/step - loss: 0.4645 - accuracy: 0.7936 - val_loss: 0.4364 - val_accuracy: 0.8058
Epoch 6/50
212/215 [============================>.] - ETA: 0s - loss: 0.4543 - accuracy: 0.7978
Epoch 6: val_accuracy did not improve from 0.81890
215/215 [==============================] - 4s 18ms/step - loss: 0.4542 - accuracy: 0.7977 - val_loss: 0.4348 - val_accuracy: 0.8163
Epoch 7/50
214/215 [============================>.] - ETA: 0s - loss: 0.4612 - accuracy: 0.7953
Epoch 7: val_accuracy did not improve from 0.81890
215/215 [==============================] - 6s 27ms/step - loss: 0.4613 - accuracy: 0.7952 - val_loss: 0.4444 - val_accuracy: 0.8071
Epoch 8/50
213/215 [==========

INFO:tensorflow:Assets written to: tweets/conv/assets


INFO:tensorflow:Assets written to: tweets/conv/assets


215/215 [==============================] - 20s 94ms/step - loss: 0.4429 - accuracy: 0.8044 - val_loss: 0.4344 - val_accuracy: 0.8255
Epoch 15/50
214/215 [============================>.] - ETA: 0s - loss: 0.4499 - accuracy: 0.8015
Epoch 15: val_accuracy did not improve from 0.82546
215/215 [==============================] - 5s 25ms/step - loss: 0.4501 - accuracy: 0.8015 - val_loss: 0.4330 - val_accuracy: 0.8084
Epoch 16/50
212/215 [============================>.] - ETA: 0s - loss: 0.4387 - accuracy: 0.8068
Epoch 16: val_accuracy did not improve from 0.82546
215/215 [==============================] - 6s 27ms/step - loss: 0.4394 - accuracy: 0.8061 - val_loss: 0.4387 - val_accuracy: 0.8058
Epoch 17/50
215/215 [==============================] - ETA: 0s - loss: 0.4411 - accuracy: 0.8053
Epoch 17: val_accuracy did not improve from 0.82546
215/215 [==============================] - 4s 18ms/step - loss: 0.4411 - accuracy: 0.8053 - val_loss: 0.4283 - val_accuracy: 0.8189
Epoch 18/50
212/215 [===

In [60]:
best_model = tf.keras.models.load_model("tweets/conv")

In [61]:
best_model.evaluate(test_dataset)

24/24 [==============================] - 1s 12ms/step - loss: 0.4344 - accuracy: 0.8255


[0.4343593716621399, 0.8254593014717102]

#### LSTM Model

In [75]:
# Create model using the Sequential API
model_3 = tf.keras.Sequential([
  sentence_encoder_layer, # take in sentences and then encode them into an embedding
  tfl.Reshape((64, 8), input_shape=(512,)),

  tfl.LSTM(32, return_sequences = True),
  tfl.LSTM(64, return_sequences = True),
  tfl.MaxPool1D(2),
  tfl.Dropout(0.6),
  tfl.BatchNormalization(),

   tfl.LSTM(128, return_sequences = True),
   tfl.LSTM(256, return_sequences = True),
  tfl.Dropout(0.6),

    tfl.GlobalAveragePooling1D(),
    tfl.Dense(128, "relu"),
    tfl.Dropout(0.7),
      tfl.BatchNormalization(),
    tfl.Dense(256, "relu"),
    tfl.Dropout(0.8),
  tfl.Dense(1, activation="sigmoid")
], name="model_3_USE")

# Compile model
model_3.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

model_3.summary()

Model: "model_3_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 reshape_17 (Reshape)        (None, 64, 8)             0         
                                                                 
 lstm_33 (LSTM)              (None, 64, 32)            5248      
                                                                 
 lstm_34 (LSTM)              (None, 64, 64)            24832     
                                                                 
 max_pooling1d_32 (MaxPoolin  (None, 32, 64)           0         
 g1D)                                                            
                                                                 
 dropout_57 (Dropout)        (None, 32, 64)            0         
                                                       

In [76]:
history_4 = model_3.fit(train_dataset,
                        epochs=10,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        # callbacks = [create_model_checkpoint("conv")]
                        )

Epoch 1/10
215/215 [==============================] - 15s 33ms/step - loss: 0.7401 - accuracy: 0.5469 - val_loss: 0.6829 - val_accuracy: 0.5656
Epoch 2/10
215/215 [==============================] - 5s 23ms/step - loss: 0.7231 - accuracy: 0.5454 - val_loss: 0.7130 - val_accuracy: 0.5656
Epoch 3/10
215/215 [==============================] - 5s 24ms/step - loss: 0.6784 - accuracy: 0.5860 - val_loss: 0.7262 - val_accuracy: 0.4475
Epoch 4/10
215/215 [==============================] - 5s 24ms/step - loss: 0.6520 - accuracy: 0.6253 - val_loss: 0.7843 - val_accuracy: 0.4357
Epoch 5/10
215/215 [==============================] - 5s 24ms/step - loss: 0.6458 - accuracy: 0.6336 - val_loss: 0.6576 - val_accuracy: 0.5984
Epoch 6/10
215/215 [==============================] - 7s 30ms/step - loss: 0.6381 - accuracy: 0.6492 - val_loss: 0.6435 - val_accuracy: 0.6745
Epoch 7/10
215/215 [==============================] - 5s 25ms/step - loss: 0.6290 - accuracy: 0.6574 - val_loss: 0.6429 - val_accuracy: 0.664

LSTM is not performing very well. Let's try birectional LSTM.

#### Bidirectional LSTM

In [102]:
# Create model using the Sequential API
model_4 = tf.keras.Sequential([
  sentence_encoder_layer, # take in sentences and then encode them into an embedding
  tfl.Reshape((64, 8), input_shape=(512,)),

  tfl.Bidirectional(tfl.LSTM(64, return_sequences = True)),
  tfl.Bidirectional(tfl.LSTM(128, return_sequences = True)),
#   tfl.Bidirectional(tfl.LSTM(256, return_sequences = True)),
#   tfl.MaxPool1D(2),
  tfl.Dropout(0.6),
#   tfl.BatchNormalization(),

   tfl.Bidirectional(tfl.LSTM(128, return_sequences = True)),
   tfl.Bidirectional(tfl.LSTM(256, return_sequences = True)),
  tfl.Dropout(0.6),

    tfl.GlobalAveragePooling1D(),
    # tfl.Dense(128, "relu"),
    # tfl.Dropout(0.5),
    # tfl.BatchNormalization(),
    # tfl.Dense(256, "relu"),
    # tfl.Dropout(0.5),

  tfl.Dense(1, activation="sigmoid")
], name="model_4_USE")

# Compile model
model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

model_4.summary()

Model: "model_4_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 reshape_29 (Reshape)        (None, 64, 8)             0         
                                                                 
 bidirectional_20 (Bidirecti  (None, 64, 128)          37376     
 onal)                                                           
                                                                 
 bidirectional_21 (Bidirecti  (None, 64, 256)          263168    
 onal)                                                           
                                                                 
 dropout_89 (Dropout)        (None, 64, 256)           0         
                                                                 
 bidirectional_22 (Bidirecti  (None, 64, 256)          

In [103]:
history_5 = model_4.fit(train_dataset,
                        epochs=10,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        # callbacks = [create_model_checkpoint("conv")]
                        )

Epoch 1/10
215/215 [==============================] - 23s 59ms/step - loss: 0.6587 - accuracy: 0.6178 - val_loss: 0.6677 - val_accuracy: 0.6417
Epoch 2/10
215/215 [==============================] - 9s 42ms/step - loss: 0.6394 - accuracy: 0.6416 - val_loss: 0.6337 - val_accuracy: 0.6706
Epoch 3/10
215/215 [==============================] - 9s 41ms/step - loss: 0.6302 - accuracy: 0.6530 - val_loss: 0.6260 - val_accuracy: 0.6654
Epoch 4/10
215/215 [==============================] - 9s 41ms/step - loss: 0.6231 - accuracy: 0.6645 - val_loss: 0.6236 - val_accuracy: 0.6706
Epoch 5/10
215/215 [==============================] - 9s 41ms/step - loss: 0.6045 - accuracy: 0.6759 - val_loss: 0.6040 - val_accuracy: 0.6982
Epoch 6/10
215/215 [==============================] - 9s 41ms/step - loss: 0.6182 - accuracy: 0.6607 - val_loss: 0.6435 - val_accuracy: 0.6850
Epoch 7/10
215/215 [==============================] - 9s 42ms/step - loss: 0.6796 - accuracy: 0.5726 - val_loss: 0.6773 - val_accuracy: 0.572

Let's fit for some more epochs

In [104]:
history_5 = model_4.fit(train_dataset,
                        epochs=50,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        callbacks = [create_model_checkpoint("bi_LSTM")]
                        )

Saving model checkpoint to: tweets/bi_LSTM
Epoch 1/50
215/215 [==============================] - ETA: 0s - loss: 0.5465 - accuracy: 0.7328
Epoch 1: val_accuracy improved from -inf to 0.74541, saving model to tweets/bi_LSTM


INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


215/215 [==============================] - 61s 282ms/step - loss: 0.5465 - accuracy: 0.7328 - val_loss: 0.5433 - val_accuracy: 0.7454
Epoch 2/50
214/215 [============================>.] - ETA: 0s - loss: 0.5349 - accuracy: 0.7404
Epoch 2: val_accuracy did not improve from 0.74541
215/215 [==============================] - 9s 43ms/step - loss: 0.5350 - accuracy: 0.7403 - val_loss: 0.5352 - val_accuracy: 0.7388
Epoch 3/50
214/215 [============================>.] - ETA: 0s - loss: 0.5281 - accuracy: 0.7449
Epoch 3: val_accuracy improved from 0.74541 to 0.75591, saving model to tweets/bi_LSTM


INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


...

INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


215/215 [==============================] - 60s 280ms/step - loss: 0.4797 - accuracy: 0.7787 - val_loss: 0.4861 - val_accuracy: 0.7822
Epoch 22/50
214/215 [============================>.] - ETA: 0s - loss: 0.4752 - accuracy: 0.7785
Epoch 22: val_accuracy did not improve from 0.78215
215/215 [==============================] - 9s 42ms/step - loss: 0.4752 - accuracy: 0.7785 - val_loss: 0.4945 - val_accuracy: 0.7690
Epoch 23/50
215/215 [==============================] - ETA: 0s - loss: 0.4753 - accuracy: 0.7829
Epoch 23: val_accuracy did not improve from 0.78215
215/215 [==============================] - 9s 41ms/step - loss: 0.4753 - accuracy: 0.7829 - val_loss: 0.4921 - val_accuracy: 0.7677
Epoch 24/50
215/215 [==============================] - ETA: 0s - loss: 0.4682 - accuracy: 0.7860
Epoch 24: val_accuracy improved from 0.78215 to 0.78346, saving model to tweets/bi_LSTM


INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


INFO:tensorflow:Assets written to: tweets/bi_LSTM/assets


215/215 [==============================] - 60s 281ms/step - loss: 0.4682 - accuracy: 0.7860 - val_loss: 0.4927 - val_accuracy: 0.7835
Epoch 25/50
215/215 [==============================] - ETA: 0s - loss: 0.4717 - accuracy: 0.7841
Epoch 25: val_accuracy did not improve from 0.78346
215/215 [==============================] - 10s 47ms/step - loss: 0.4717 - accuracy: 0.7841 - val_loss: 0.4798 - val_accuracy: 0.7808
Epoch 26/50
214/215 [============================>.] - ETA: 0s - loss: 0.4720 - accuracy: 0.7826
Epoch 26: val_accuracy did not improve from 0.78346
215/215 [==============================] - 9s 41ms/step - loss: 0.4719 - accuracy: 0.7826 - val_loss: 0.4911 - val_accuracy: 0.7782
Epoch 27/50
215/215 [==============================] - ETA: 0s - loss: 0.4674 - accuracy: 0.7883
Epoch 27: val_accuracy did not improve from 0.78346
215/215 [==============================] - 9s 42ms/step - loss: 0.4674 - accuracy: 0.7883 - val_loss: 0.4938 - val_accuracy: 0.7612
Epoch 28/50
214/215 [=

#### GRU Unit

In [95]:
# Create model using the Sequential API
model_4 = tf.keras.Sequential([
  sentence_encoder_layer, # take in sentences and then encode them into an embedding
  tfl.Reshape((32, 16), input_shape=(512,)),

  tfl.GRU(32, return_sequences = True),
  tfl.GRU(64, return_sequences = True),
  tfl.MaxPool1D(2),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),

   tfl.GRU(128, return_sequences = True),
   tfl.GRU(256, return_sequences = True),
   tfl.Dropout(0.8),

    tfl.GlobalAveragePooling1D(),
    # tfl.Dense(128, "relu"),
    # tfl.Dropout(0.7),
    # tfl.BatchNormalization(),
    # tfl.Dense(256, "relu"),
    # tfl.Dropout(0.8),
  tfl.Dense(1, activation="sigmoid")
], name="model_4_USE")

# Compile model
model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

model_4.summary()

Model: "model_4_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 reshape_26 (Reshape)        (None, 32, 16)            0         
                                                                 
 gru_16 (GRU)                (None, 32, 32)            4800      
                                                                 
 gru_17 (GRU)                (None, 32, 64)            18816     
                                                                 
 max_pooling1d_38 (MaxPoolin  (None, 16, 64)           0         
 g1D)                                                            
                                                                 
 dropout_83 (Dropout)        (None, 16, 64)            0         
                                                       

In [96]:
history_5 = model_4.fit(train_dataset,
                        epochs=10,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        # callbacks = [create_model_checkpoint("conv")]
                        )

Epoch 1/10
215/215 [==============================] - 13s 27ms/step - loss: 0.6322 - accuracy: 0.6474 - val_loss: 0.6650 - val_accuracy: 0.6273
Epoch 2/10
215/215 [==============================] - 4s 20ms/step - loss: 0.6063 - accuracy: 0.6766 - val_loss: 0.6379 - val_accuracy: 0.6417
Epoch 3/10
215/215 [==============================] - 4s 20ms/step - loss: 0.5795 - accuracy: 0.7016 - val_loss: 0.8994 - val_accuracy: 0.5656
Epoch 4/10
215/215 [==============================] - 4s 20ms/step - loss: 0.5743 - accuracy: 0.7091 - val_loss: 0.6543 - val_accuracy: 0.5971
Epoch 5/10
215/215 [==============================] - 4s 19ms/step - loss: 0.5537 - accuracy: 0.7280 - val_loss: 0.6966 - val_accuracy: 0.6430
Epoch 6/10
215/215 [==============================] - 4s 19ms/step - loss: 0.5502 - accuracy: 0.7293 - val_loss: 0.5797 - val_accuracy: 0.7126
Epoch 7/10
215/215 [==============================] - 4s 19ms/step - loss: 0.5434 - accuracy: 0.7366 - val_loss: 0.5244 - val_accuracy: 0.749

Interesting! Let's fit for some more epochs.

In [97]:
history_5 = model_4.fit(train_dataset,
                        epochs=50,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        callbacks = [create_model_checkpoint("GRU")]
                        )

Saving model checkpoint to: tweets/GRU
Epoch 1/50
214/215 [============================>.] - ETA: 0s - loss: 0.5128 - accuracy: 0.7605
Epoch 1: val_accuracy improved from -inf to 0.76640, saving model to tweets/GRU


INFO:tensorflow:Assets written to: tweets/GRU/assets


INFO:tensorflow:Assets written to: tweets/GRU/assets


215/215 [==============================] - 25s 116ms/step - loss: 0.5128 - accuracy: 0.7604 - val_loss: 0.5063 - val_accuracy: 0.7664
Epoch 2/50
214/215 [============================>.] - ETA: 0s - loss: 0.5088 - accuracy: 0.7629
Epoch 2: val_accuracy improved from 0.76640 to 0.76903, saving model to tweets/GRU


INFO:tensorflow:Assets written to: tweets/GRU/assets


...

INFO:tensorflow:Assets written to: tweets/GRU/assets


215/215 [==============================] - 25s 116ms/step - loss: 0.4483 - accuracy: 0.8026 - val_loss: 0.4616 - val_accuracy: 0.7992
Epoch 23/50
213/215 [============================>.] - ETA: 0s - loss: 0.4463 - accuracy: 0.7990
Epoch 23: val_accuracy did not improve from 0.79921
215/215 [==============================] - 4s 20ms/step - loss: 0.4462 - accuracy: 0.7990 - val_loss: 0.4680 - val_accuracy: 0.7887
Epoch 24/50
214/215 [============================>.] - ETA: 0s - loss: 0.4383 - accuracy: 0.8056
Epoch 24: val_accuracy did not improve from 0.79921
215/215 [==============================] - 4s 20ms/step - loss: 0.4383 - accuracy: 0.8057 - val_loss: 0.4577 - val_accuracy: 0.7874
Epoch 25/50
214/215 [============================>.] - ETA: 0s - loss: 0.4424 - accuracy: 0.8015
Epoch 25: val_accuracy did not improve from 0.79921
215/215 [==============================] - 4s 20ms/step - loss: 0.4424 - accuracy: 0.8015 - val_loss: 0.4585 - val_accuracy: 0.7874
Epoch 26/50
214/215 [==

INFO:tensorflow:Assets written to: tweets/GRU/assets


INFO:tensorflow:Assets written to: tweets/GRU/assets


215/215 [==============================] - 24s 113ms/step - loss: 0.4188 - accuracy: 0.8128 - val_loss: 0.4658 - val_accuracy: 0.8097
Epoch 36/50
212/215 [============================>.] - ETA: 0s - loss: 0.4191 - accuracy: 0.8144
Epoch 36: val_accuracy did not improve from 0.80971
215/215 [==============================] - 4s 19ms/step - loss: 0.4187 - accuracy: 0.8145 - val_loss: 0.4601 - val_accuracy: 0.7887
Epoch 37/50
213/215 [============================>.] - ETA: 0s - loss: 0.4141 - accuracy: 0.8150
Epoch 37: val_accuracy did not improve from 0.80971
215/215 [==============================] - 4s 20ms/step - loss: 0.4139 - accuracy: 0.8152 - val_loss: 0.5211 - val_accuracy: 0.7717
Epoch 38/50
214/215 [============================>.] - ETA: 0s - loss: 0.4110 - accuracy: 0.8164
Epoch 38: val_accuracy did not improve from 0.80971
215/215 [==============================] - 4s 20ms/step - loss: 0.4109 - accuracy: 0.8165 - val_loss: 0.5054 - val_accuracy: 0.7887
Epoch 39/50
214/215 [==

In [99]:
best_gru_model = tf.keras.models.load_model("/content/tweets/GRU")
best_gru_model.evaluate(test_dataset)

24/24 [==============================] - 2s 12ms/step - loss: 0.4658 - accuracy: 0.8097


[0.46581578254699707, 0.8097112774848938]

#### Final Model

Out of all these, the Conv model is performing the best. We'll use this as final model.

In [108]:
# Create model using the Sequential API
model_f = tf.keras.Sequential([
  sentence_encoder_layer, # take in sentences and then encode them into an embedding
  tfl.Reshape((64, 8), input_shape=(512,)),

  tfl.Conv1D(32, 5, padding="valid"),
  tfl.Conv1D(64, 5, padding="valid"),
   tfl.MaxPool1D(2),
  tfl.Dropout(0.7),
  tfl.BatchNormalization(),

  tfl.Conv1D(128, 5, padding="valid"),
  tfl.Conv1D(256, 5, padding="valid"),
   tfl.MaxPool1D(2),
  tfl.Dropout(0.7),
  tfl.BatchNormalization(),

  tfl.Conv1D(256, 5, padding="valid"),
  tfl.Conv1D(512, 5, padding="valid"),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),

  tfl.GlobalAveragePooling1D(),

  tfl.Dense(128, activation="relu"),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),
  tfl.Dense(256, activation="relu"),
  tfl.Dropout(0.8),
  tfl.BatchNormalization(),
  tfl.Dense(1, activation="sigmoid")
], name="model_2_USE")

# Compile model
model_f.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

model_f.summary()

Model: "model_2_USE"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 USE (KerasLayer)            (None, 512)               256797824 
                                                                 
 reshape_32 (Reshape)        (None, 64, 8)             0         
                                                                 
 conv1d_55 (Conv1D)          (None, 60, 32)            1312      
                                                                 
 conv1d_56 (Conv1D)          (None, 56, 64)            10304     
                                                                 
 max_pooling1d_45 (MaxPoolin  (None, 28, 64)           0         
 g1D)                                                            
                                                                 
 dropout_100 (Dropout)       (None, 28, 64)            0         
                                                       

In [110]:
history_f = model_f.fit(train_dataset,
                        epochs=100,
                        steps_per_epoch = len(train_dataset),
                        validation_data = test_dataset,
                        validation_steps = len(test_dataset),
                        callbacks = [create_model_checkpoint("final")]
                        )

Saving model checkpoint to: tweets/final
Epoch 1/100
214/215 [============================>.] - ETA: 0s - loss: 0.5639 - accuracy: 0.7279
Epoch 1: val_accuracy improved from -inf to 0.78084, saving model to tweets/final


INFO:tensorflow:Assets written to: tweets/final/assets


INFO:tensorflow:Assets written to: tweets/final/assets


215/215 [==============================] - 16s 75ms/step - loss: 0.5640 - accuracy: 0.7279 - val_loss: 0.4979 - val_accuracy: 0.7808
Epoch 2/100
213/215 [============================>.] - ETA: 0s - loss: 0.5547 - accuracy: 0.7405
Epoch 2: val_accuracy improved from 0.78084 to 0.78478, saving model to tweets/final


INFO:tensorflow:Assets written to: tweets/final/assets


...

INFO:tensorflow:Assets written to: tweets/final/assets


215/215 [==============================] - 15s 68ms/step - loss: 0.4712 - accuracy: 0.7949 - val_loss: 0.4491 - val_accuracy: 0.8202
Epoch 74/100
214/215 [============================>.] - ETA: 0s - loss: 0.4693 - accuracy: 0.7969
Epoch 74: val_accuracy did not improve from 0.82021
215/215 [==============================] - 4s 18ms/step - loss: 0.4694 - accuracy: 0.7968 - val_loss: 0.4487 - val_accuracy: 0.8097
Epoch 75/100
213/215 [============================>.] - ETA: 0s - loss: 0.4674 - accuracy: 0.8005
Epoch 75: val_accuracy did not improve from 0.82021
215/215 [==============================] - 4s 17ms/step - loss: 0.4677 - accuracy: 0.8003 - val_loss: 0.4475 - val_accuracy: 0.8123
Epoch 76/100
214/215 [============================>.] - ETA: 0s - loss: 0.4778 - accuracy: 0.7950
Epoch 76: val_accuracy did not improve from 0.82021
215/215 [==============================] - 4s 17ms/step - loss: 0.4779 - accuracy: 0.7949 - val_loss: 0.4433 - val_accuracy: 0.8097
Epoch 77/100
214/215 

INFO:tensorflow:Assets written to: tweets/final/assets


INFO:tensorflow:Assets written to: tweets/final/assets


215/215 [==============================] - 15s 68ms/step - loss: 0.4785 - accuracy: 0.7936 - val_loss: 0.4468 - val_accuracy: 0.8215
Epoch 79/100
214/215 [============================>.] - ETA: 0s - loss: 0.4732 - accuracy: 0.7951
Epoch 79: val_accuracy did not improve from 0.82152
215/215 [==============================] - 4s 17ms/step - loss: 0.4733 - accuracy: 0.7950 - val_loss: 0.4521 - val_accuracy: 0.7795
Epoch 80/100
215/215 [==============================] - ETA: 0s - loss: 0.4780 - accuracy: 0.7953
Epoch 80: val_accuracy did not improve from 0.82152
215/215 [==============================] - 4s 18ms/step - loss: 0.4780 - accuracy: 0.7953 - val_loss: 0.4423 - val_accuracy: 0.8084
Epoch 81/100
212/215 [============================>.] - ETA: 0s - loss: 0.4730 - accuracy: 0.7951
Epoch 81: val_accuracy did not improve from 0.82152
215/215 [==============================] - 4s 18ms/step - loss: 0.4737 - accuracy: 0.7946 - val_loss: 0.4432 - val_accuracy: 0.8136
Epoch 82/100
212/215 

In [114]:
best_model = tf.keras.models.load_model("tweets/final")
best_model.evaluate(test_dataset)

24/24 [==============================] - 1s 11ms/step - loss: 0.4468 - accuracy: 0.8215


[0.44681739807128906, 0.8215222954750061]

In [115]:
best_model.evaluate(train_dataset)

215/215 [==============================] - 4s 18ms/step - loss: 0.4346 - accuracy: 0.8212


[0.4345894455909729, 0.8211678862571716]

In [121]:
preds = best_model.predict(test_text)

In [128]:
preds_final = tf.squeeze(tf.round(preds))
preds_final

<tf.Tensor: shape=(3263,), dtype=float32, numpy=array([1., 1., 1., ..., 1., 1., 1.], dtype=float32)>

In [129]:
sub = pd.DataFrame({"id": id, "target": preds_final})
sub

,id,target
0,0,1.0
1,2,1.0
2,3,1.0
3,9,1.0
4,11,1.0
...,...,...
3258,10861,1.0
3259,10865,1.0
3260,10868,1.0
3261,10874,1.0


In [130]:
sub["target"] = sub["target"].apply(lambda x:int(x))

In [132]:
sub.to_csv("sub.csv", index=False)

In [ ]:
def get_submission_file(model, id=id, file_name="sub.csv"):
    preds = model.predict(test_text)
    preds_final = tf.squeeze(tf.round(preds))
    sub = pd.DataFrame({"id": id, "target": preds_final})
    sub["target"] = sub["target"].apply(lambda x:int(x))
    sub.to_csv(file_name, index=False)
    return sub

In [134]:
!zip -r /content/models.zip /content/tweets/final

  adding: content/tweets/final/ (stored 0%)
  adding: content/tweets/final/variables/ (stored 0%)
  adding: content/tweets/final/variables/variables.data-00000-of-00001 (deflated 7%)
  adding: content/tweets/final/variables/variables.index (deflated 78%)
  adding: content/tweets/final/keras_metadata.pb (deflated 94%)
  adding: content/tweets/final/saved_model.pb (deflated 68%)
  adding: content/tweets/final/assets/ (stored 0%)


In [ ]:
from google.colab import files
files.download("/content/file.zip")